#### 一. 什么是 Tokenizer

大模型（LLM）并不能直接理解自然语言。

对于自然语言中的一句话，想要让 LLM 理解，首先需要将自然语言中的一句话划分为很多块，然后需要将每块的“词”变成其专属的“数字ID”

这就是分词器（Tokenizer）需要做的事 ———— 把自然语言转换成大模型能看懂的数字

注意：
1. 动态决定颗粒度: 常见的组合 → 打包成一个整体，罕见的组合 → 拆成多个小块
2. Tokenizer 的实现方式有很多，每种方式中对于同一个“词”的表示大概率不同，即不同方式得到的“数字ID”不同
3. Tokenizer 对原始句子、词、单词进行划分时，最好是符合语言语义的
> 以划分incredible为例——['i','n','cr','e','dible']就过于”碎片化“了，
> ['incredible']过于”笼统“，理想的划分是['in','credible']这样能在不同单词中复用，
> 既控制了token总数，又提升了embedding的映射能力。

#### 二、训练分词器的步骤


**STEP-1. 准备语料**

1. 语料丰富性
    
    语料尽可能多样
    - 不同类型: 小说、散文、诗歌等
    - 多种语言: 英文、中文、日文等
    
    多语言或混合语言场景中问题：

    因为不同语言的语料占比不同，可能会把较少语料中的常出现的词，切分的过细

    解决办法：
    - 增加语料较少的语言的语料，尽量让多语料平均
    - 调整比例
    - 上采样语料较少的语言的语料
    - 下采样语料较多的语言的语料

2. 原始文本的清洗和标准化
    
    主要操作：
    - 去除或屏蔽无关元数据
    - 修改或删除乱码或异常字符
    - 统一编码
    - 对敏感信息进行脱敏处理与合规检验
    
    > 为什么要进行数据脱敏？
    > 
    > 1. 保护隐私
    > 2. 这些隐私数据一般都属于低频噪声，会干扰分词算法
    >
    > > 这样可以有效降低语料中无效的多样性，使分词器更专注于建模具有统计规律的语言模式，从而提升词表的利用效率、一致性。从香农信息论的角度来看，数据脱敏可以视为一种结构化的去噪过程——通过压缩或减少高熵但低语义价值的信号（如具体身份信息），提高语料中有效信号的占比，可以协助LLM在后面的训练过程更倾向于学习可复用的语义结构，而非记忆偶然出现的实例细节。
    > >
    > > Tips：高熵是指在给定上下文条件下，概率分布相对分散，从而具有较高的不确定性和预测困难度较高。
    > 
    > 注意平衡好敏感数据和关键语义信息

3. 保留一部分语料做测试集，用于评估分词器性能 

**STEP-2. 预分词**

